In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Версия Python:", sys.version)
print("Путь к интерпретатору:", sys.executable)
print('numpy :', np.__version__)
print('pandas:', pd.__version__)
print('seaborn:', sns.__version__)

Проверим расположение clients.csv и transactions.csv

In [ ]:
# Проверка наличия данных
from pathlib import Path

DATA_DIR = Path("data/raw")

CLIENTS_PATH = DATA_DIR / "clients.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"

print("Папка:", DATA_DIR.absolute())
print("clients.csv:", CLIENTS_PATH.is_file())
print("transactions.csv:", TRANSACTIONS_PATH.is_file())

## 1. Аудит качества данных

## 1.1 Наивная загрузка
Читаем данные без разделителей и типов

In [ ]:
if CLIENTS_PATH.is_file() and TRANSACTIONS_PATH.is_file():
    df_clients_naive = pd.read_csv(CLIENTS_PATH, encoding='utf-8')
    df_transactions_naive = pd.read_csv(TRANSACTIONS_PATH, encoding='cp1251')
    print("\nclients.shape:", df_clients_naive.shape)
    print("transactions.shape:", df_transactions_naive.shape)
    
    print(df_clients_naive.dtypes.value_counts())
    print(df_transactions_naive.dtypes.value_counts())
    
    print(df_clients_naive.head(3))
    print(df_transactions_naive.head(5))
else:
    print("Files not found")
    

Вывод: 
- clients.csv весь прочитан в таблицу (5702, 16)
- transactions.csv прочитан некорректно: получен один столбец вместо отдельных полей, видно, что разделителем является ';'

In [ ]:
# Типы всех столбцов
print("\n Типы данных всех столбцов:")
print(df_clients_naive.dtypes)

Вывод: типы требуют уточнения и не совсем корректны:
 - даты(birth_date, registration_date, card_open_date, app_last_login) представлены str, а не спец типом даты
  - satisfaction_score содержит целые оценки, а не дробные

In [ ]:
# Количество пропусков (NaN) в каждом столбце
print("\n Пропуски (NaN) в каждом столбце:")
print(df_clients_naive.isnull().sum())

## 1.2 Правильная загрузка clients
Зададим типы полей и даты разберём отдельно форматам источника


In [ ]:
df_clients = pd.read_csv(
    CLIENTS_PATH,
    encoding='utf-8',
    dtype={
        'client_id': 'str',
        'city': 'str',
        'branch_code': 'str',  # Читаем ка строку, так как в начале есть нули, которые при преобразовании в int станут незначащими и исчезнут
        'gender': 'category',
        'segment': 'category',
        'loyalty_tier': 'category',
        'card_type': 'category',
        'income_source': 'category',
        'has_mobile_app': 'category',
        'marketing_opt_in': 'category',
        'satisfaction_score': 'Int64',    # Это порядковая шкала, мне удобно Int так как порядок тогда уже задан 
        'monthly_income': 'float64',
    }
)
df_clients['birth_date'] = pd.to_datetime(
    df_clients['birth_date'], format='%Y-%m-%d', errors='coerce'
)
df_clients['registration_date'] = pd.to_datetime(
    df_clients['registration_date'], format='%d.%m.%Y', errors='coerce'
)
df_clients['card_open_date'] = pd.to_datetime(
    df_clients['card_open_date'], format='%d.%m.%Y', errors='coerce'
)
df_clients['app_last_login'] = pd.to_datetime(
    df_clients['app_last_login'], format='%Y-%m', errors='coerce'
)


print("clients.shape:", df_clients.shape)

## 1.3 Категории и обозначения отсутсвующих сведений
## Аудит и нормализация категориальные данные

In [ ]:
# Список столбцов для анализа
cat_cols = [
    'gender', 'city', 'segment', 'loyalty_tier', 'card_type',
    'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in'
]

# Частота категорий в каждом столбце
for col in cat_cols:
    print(f"\n Столбец: {col}")
    print(df_clients[col].value_counts(dropna=False))  # dropna=False — показывает NaN тоже

Выводы: 
- Есть разные написания одних и тех же городов
- Нет мобильного приложения(has_mobile_app) у 2140 и у 2140 клиентов NaT в app_last_login, надо проверить одни и те же ли это клиенты
- 1549 клиентов не указали источник зарплаты. Нужно посмотреть опять же связи с другими признаками
- 648 клиента отказались от участия маркетинговой коммуникации, поэтому они не будут включены ни в одну из групп, так как они не интересны
- 3416 пропуска в satisfaction_score нельзя ничем заменить. Далее проверим есть ли связи с другими признаками

In [ ]:
# Приведём все города к одному написанию
import re

def normalize_city(city):
    """Brings the city to its standard appearance"""
    if pd.isna(city):
        return city
    
    city = city.strip()  # убираем пробелы
    # Убираем "г. " или "город " в начале
    city = re.sub(r'^(г\.|город)\s*', '', city, flags=re.IGNORECASE)
    city = city.title() # Один регистр
    
    abbreviations = {
        'Спб': 'Санкт-Петербург',
        'Санкт Петербург': 'Санкт-Петербург',
        'Н.Новгород': 'Нижний Новгород',
        'Н. Новгород': 'Нижний Новгород',
        'Ростов-На-Дону': 'Ростов-на-Дону',
    }
    
    return abbreviations.get(city, city)

df_clients['city'] = df_clients['city'].apply(normalize_city)

print(df_clients['city'].value_counts(dropna=False))

Вывод: разные написания одного и того же города привели к одному названию.

In [ ]:
# Проверяет соответствие отсутствия даты последнего входа и отсутствие мобильного приложения 
mask_anomaly = (
    (df_clients['has_mobile_app'] == 'Нет') & (df_clients['app_last_login'].notna())  
    |
    (df_clients['has_mobile_app'] == 'Да') & (df_clients['app_last_login'].isna())    
)

df_anomalies = df_clients[mask_anomaly].copy()

print(f"Найдено аномальных записей: {len(df_anomalies)}")
print(f"Из {len(df_clients)} всего ({len(df_anomalies)/len(df_clients)*100:.2f}%)")

# Смотрим на аномалии
display(df_anomalies[['client_id', 'has_mobile_app', 'app_last_login']].head(20))

Вывод: даты последнего входа не указаны только у людей, у которых нет приложения. Данный пропуск ничем не заменяем

## 1.4 Повторные записи
Разбираем столбец client_id

In [ ]:
# Проверим уникальность клиентов
n_clients_raw = len(df_clients)
n_unique_clients_raw = df_clients['client_id'].nunique()

print('Всего строк:', n_clients_raw)
print('Уникальных client_id:', n_unique_clients_raw)
print('Пропусков client_id:', df_clients['client_id'].isna().sum())
print('Строк с повторяющимся client_id:',
      df_clients['client_id'].duplicated(keep=False).sum())

fully_duplicated_rows = df_clients.duplicated().sum()
print('Лишних идентичных записей:', fully_duplicated_rows)

Вывод: обнаружено 40 полный дубликатов, я считаю, что нужно удалить дубликаты, так как очень маловероятно, что человек открыл две абсолютно одинаковые карты в один день(сейчас это никак нельзя проверить точно).
 Остальные частичные дубликаты с одинаковым client_id нужно проверить: какие признаки отличаются(т.е. возможно ли что это 2 или более карт у одного клиента)

In [ ]:
# Удалим полные дубликаты
n_before = len(df_clients)
df_clients = df_clients.drop_duplicates().copy()
n_full_removed = n_before - len(df_clients)

print('Удалено полных дубликатов:', n_full_removed)
print('Осталось строк:', len(df_clients))

In [ ]:
# Проверяем все неидентичные записи клиентов с повторяющимся ID
conflict_mask = df_clients['client_id'].duplicated(keep=False)

clients_conflicts = (
    df_clients.loc[conflict_mask]
    .sort_values('client_id')
    .copy()
)

print('Конфликтующих строк:', len(clients_conflicts))
print('Клиентов с конфликтом:',
      clients_conflicts['client_id'].nunique())

# Количество разных значений каждого поля внутри client_id
different = (
    clients_conflicts
    .groupby('client_id', observed=True)
    .nunique(dropna=False)
    .gt(1)
)

# Число клиентов с расхождениями по каждому полю
conflict_counts = different.sum()

display(
    conflict_counts[conflict_counts > 0]
    .to_frame('n_clients_with_conflict')
)

conflict_cols = different.columns[different.any()].tolist()
display(clients_conflicts[['client_id'] + conflict_cols])

Вывод: всего 30 неидентичных строк с одинаковым client_id, отличия: 13 по городам и 15 по сегментам, так как остальные поля, включая тип карты и дату открытия, совпадают значит нет повода считать эти строки как отдельные строки.
Так как нет вариантов определить какая запись актуальна, то эти 0.27% от 5647 уникальных клиентов исключим из основной выборки.

In [ ]:
# сохраняем id конфликтующих клиентов для аудита
conflict_ids = clients_conflicts['client_id'].unique()

# Удаляем всех конфликтующих клиентов
df_clients = df_clients.loc[
    ~df_clients['client_id'].isin(conflict_ids)
].copy()

assert df_clients['client_id'].notna().all()
assert df_clients['client_id'].is_unique

print('Осталось строк:', len(df_clients))
print('Уникальных client_id:', df_clients['client_id'].nunique())
print('Исключено уникальных клиентов:', len(conflict_ids))


Вывод: в таблице осталось 5632 строчки с уникальными client_id 

## 1.5 Возраст и согласованность дат
Справочник представлен на момент 30.06.2025

In [ ]:
REFERENCE_DATE = pd.Timestamp('2025-06-30')

# Проверим типы
date_cols = [
    'birth_date',
    'registration_date',
    'card_open_date',
    'app_last_login',
]

print(df_clients[date_cols].dtypes)

In [ ]:
def full_years(birth_date, event_date):
    """
    Calculates the age in full years between the date of birth and the event
    """
    # Проверяем наступил ли день рождения в году события
    birthday_not_reached = (
        (event_date.dt.month < birth_date.dt.month)
        | ((event_date.dt.month == birth_date.dt.month)
           & (event_date.dt.day < birth_date.dt.day))
    )
    
    years = (
        event_date.dt.year - birth_date.dt.year
        - birthday_not_reached.astype('int64')
    )
    
    valid = birth_date.notna() & event_date.notna()
    return years.where(valid).astype('Int64')

# Создаём Series с контрольной датой для каждого клиента
reference_dates = pd.Series(REFERENCE_DATE, index=df_clients.index)

df_clients['age'] = full_years(df_clients['birth_date'], reference_dates)
df_clients['age_at_registration'] = full_years(
    df_clients['birth_date'], df_clients['registration_date'],
)
df_clients['age_at_card_open'] = full_years(
    df_clients['birth_date'], df_clients['card_open_date'],
)

In [ ]:
display(df_clients['age'].describe())

age_cols = [
    'client_id', 'birth_date', 'age',
    'registration_date', 'age_at_registration',
    'card_open_date', 'age_at_card_open', 'card_type',
]

# Самые молодые
display(df_clients.sort_values('age')[age_cols].head(10))
# Самые пожилые
display(df_clients.sort_values('age', ascending=False)[age_cols].head(10))


Вывод: возраст клиентов составляет от 23 до 88 лет. Эти возраста на данный момент полностью подходят для любых операций. Но нужно ещё проверить их возраст на момент регистрации и открытия карты

In [ ]:
# Зафиксируем возрастные рамки для различных 
age_flags = pd.DataFrame({
    'birth_in_future': df_clients['birth_date'] > REFERENCE_DATE,
    'age_over_110': df_clients['age'] > 110,
    'age_under_14': df_clients['age'] < 14,
    'card_open_under_14': df_clients['age_at_card_open'] < 14,
    'credit_card_under_18': (
        df_clients['card_type'].eq('Кредитная')
        & df_clients['age_at_card_open'].lt(18)
    ),
}).fillna(False)

display(age_flags.sum().to_frame('n_rows'))
display(df_clients.loc[age_flags.any(axis=1), age_cols])

Вывод: у трёх клиентов на данный момент тип карты указан как "кредитная", а возраст при открытии этой карты составляет 16-17 лет. Это может свидетельствовать либо о несогласованности атрибутов, либо о правонарушении, либо об эмансипации

In [ ]:
date_flags = pd.DataFrame({
    'registration_before_birth': (
        df_clients['registration_date'] < df_clients['birth_date']
    ),
    'card_before_birth': (
        df_clients['card_open_date'] < df_clients['birth_date']
    ),
    'registration_in_future': (
        df_clients['registration_date'] > REFERENCE_DATE
    ),
    'card_open_in_future': (
        df_clients['card_open_date'] > REFERENCE_DATE
    ),
    'card_before_registration': ( # Карта открыта ДО регистрации 
        df_clients['card_open_date'] < df_clients['registration_date']
    ),
    'login_after_snapshot_month': ( #  Последний вход в приложение был ПОСЛЕ контрольного месяца
        df_clients['app_last_login'].dt.to_period('M')
        > REFERENCE_DATE.to_period('M')
    ),
    'login_before_registration_month': (# Последний вход в приложение был ДО месяца регистрации
        df_clients['app_last_login'].dt.to_period('M')
        < df_clients['registration_date'].dt.to_period('M')
    ),
}).fillna(False)

display(date_flags.sum().to_frame('n_rows'))

display(df_clients.loc[
    date_flags.any(axis=1),
    ['client_id', 'birth_date', 'registration_date',
     'card_open_date', 'has_mobile_app', 'app_last_login']
])

Вывод: у 15 клиентов месяц последнего входа предшествует месяцу регистрации. Ничего не заменяем просто запоминаем, что эта дата не всегда может показывать точно активность

## 1.6 Проверка ежемесячного дохода

In [ ]:
display(df_clients['monthly_income'].describe())

print(f"\nПропусков (NaN): {df_clients['monthly_income'].isna().sum()}")
print(f"Доля пропусков: {df_clients['monthly_income'].isna().mean() * 100:.2f}%")

# Отрицательные значения и нули
print(f"\nОтрицательных значений: {(df_clients['monthly_income'] < 0).sum()}")
print(f"Нулевых значений: {(df_clients['monthly_income'] == 0).sum()}")

# Низкие положительные доходы: проверить
low_positive = (
    df_clients['monthly_income'].gt(0)
    & df_clients['monthly_income'].lt(10_000)
)

print(f"Положительных доходов меньше 10 000 ₽: {low_positive.sum()}")

# Крупные доходы: проверить
print(f"Больше 1 000 000 ₽: {(df_clients['monthly_income'] > 1_000_000).sum()}")
print(f"Больше 10 000 000 ₽: {(df_clients['monthly_income'] > 10_000_000).sum()}")

# Квантили (показывают распределение)
print("\n=== Квантили ===")
display(df_clients['monthly_income'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Топ-10 самых высоких доходов (проверка на аномалии)
print("\n=== Топ-10 максимальных значений ===")
display(
    df_clients.nlargest(10, 'monthly_income')[
        ['client_id', 'monthly_income', 'segment', 'loyalty_tier', 'city']
    ]
)

# Топ-10 самых низких (не нулевых) доходов
print("\n=== Топ-10 минимальных значений (не нулевых) ===")
non_zero = df_clients[df_clients['monthly_income'] > 0]
display(
    non_zero.nsmallest(10, 'monthly_income')[
        ['client_id', 'monthly_income', 'segment', 'loyalty_tier', 'city']
    ]
)

In [ ]:
negative_income = df_clients['monthly_income'].lt(0)

print('Частоты отрицательных значений:')
display(
    df_clients.loc[negative_income, 'monthly_income']
    .value_counts(dropna=False)
    .rename_axis('monthly_income')
    .to_frame('n_clients')
)

print('Клиенты с отрицательным доходом:')
display(
    df_clients.loc[
        negative_income,
        ['client_id', 'monthly_income', 'income_source',
         'segment', 'loyalty_tier', 'city']
    ].sort_values('monthly_income')
)

In [ ]:
# Сохраняем исходные значения для проверки и журнала очистки
if 'monthly_income_raw' not in df_clients.columns:
    df_clients['monthly_income_raw'] = df_clients['monthly_income'].copy()

# Отмечаем клиентов, у которых исходный доход равен -999
df_clients['income_invalid'] = df_clients['monthly_income_raw'].eq(-999)

# Заменяем -999 на пропуск; остальные значения не меняем
df_clients['monthly_income'] = (
    df_clients['monthly_income'].replace(-999, np.nan)
)

print('Значений -999 в исходных данных:',
      df_clients['income_invalid'].sum())

print('Пропусков после обработки:',
      df_clients['monthly_income'].isna().sum())

print('Доля пропусков после обработки:',
      f"{df_clients['monthly_income'].isna().mean():.2%}")

# Пересчитываем статистику уже без -999
display(df_clients['monthly_income'].describe())

Вывод: обнаружено 60 значений -999, можно предположить, что это техническое отсутствие сведений, поэтому заменили на NaN, чтобы не смещать распределение. Сейчас количество пропусков 10.87% выборки

## 1.7 Исследование отсутствующих сведений
Для более удобного дальнейшего анализа добавим столбцы: 
 - income_source_missing(источник дохода не указан)
 - score_missing (отсутствует оценка)
 - income_missing (ежемесячных доход не указан)

In [ ]:
# Приведём к единому виду income_source
source_text = (
    df_clients['income_source']
    .astype('string')
    .str.strip()  
    .str.casefold()  # нижний регистр
)

# отсутствует источник дохода 
df_clients['income_source_missing'] = (
    source_text.isna() | source_text.isin(['', 'не указан'])
)
# отсутствует оценка
df_clients['score_missing'] = df_clients['satisfaction_score'].isna()
# отсутствует ежемесячный доход
df_clients['income_missing'] = df_clients['monthly_income'].isna()

flags = ['income_missing', 'income_source_missing', 'score_missing']
flag_labels = {
    'income_missing': 'Доход',
    'income_source_missing': 'Источник дохода',
    'score_missing': 'Оценка удовлетворённости',
}
missing_summary = pd.DataFrame({
    'Нет сведений, клиентов': df_clients[flags].sum(),
    'Доля, %': df_clients[flags].mean().mul(100),
}).rename(index=flag_labels)
display(missing_summary.round(2))

def missing_by(group, flag):
    result = (
        df_clients.groupby(group, observed=True, dropna=False)[flag]
        .agg(n_clients='size', n_missing='sum', missing_pct='mean')
    )
    result['missing_pct'] = result['missing_pct'].mul(100)
    return result

Вывод: из 5632 клиентов источник дохода не указан у 1536 — 27,27%.
Оценка удовлетворённости отсутствует у 3376 — 59,94%.
Значение дохода отсутствует у 552 — 9,80%.

## 1.7.1 Сопоставим полноту всех трёх показателей в единой таблице и на тепловой карте

In [ ]:
segment_counts = df_clients.groupby('segment', observed=True, dropna=False).size()
# Считаем долю пропусков (%)
segment_rates = (
    df_clients.groupby('segment', observed=True, dropna=False)[flags]
    .mean().mul(100)
)

segment_order = [
    x for x in ['Массовый', 'Зарплатный', 'Премиальный', 'Private']
    if x in segment_rates.index
]
segment_order += [x for x in segment_rates.index if x not in segment_order]
# Переупорядочиваем данные согласно заданному порядку
segment_rates = segment_rates.reindex(segment_order)
segment_counts = segment_counts.reindex(segment_order)

segment_table = segment_rates.rename(columns=flag_labels).copy()
segment_table.insert(0, 'Клиентов', segment_counts)
display(segment_table.round(1))

# Тепловая карта
heatmap_data = segment_rates.rename(columns={
    'income_missing': 'Нет дохода',
    'income_source_missing': 'Нет источника',
    'score_missing': 'Нет оценки',
}).copy()
heatmap_data.index = [
    f'{segment} (n={segment_counts.loc[segment]})'
    for segment in segment_rates.index
]
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(
    heatmap_data, annot=True, fmt='.1f', cmap='Blues',
    vmin=0, vmax=100, linewidths=0.5,
    cbar_kws={'label': 'Пропуски внутри сегмента, %'}, ax=ax,
)
ax.set(title='Полнота данных по сегментам', xlabel='', ylabel='')
ax.tick_params(axis='both', labelrotation=0)
plt.tight_layout()
plt.show()

Вывод: в массовом сегменте клиенты больше всего скрывают информацию о доходах и реже остальных оставляли оценку.

## 1.7.2 Связь суммы дохода с его источником 

In [ ]:
income_by_source = missing_by('income_source', 'income_missing')
display(income_by_source.round(1))

source_plot = income_by_source.sort_values('missing_pct')
source_labels = [
    f'{name} (n={int(row.n_clients)})'
    for name, row in source_plot.iterrows()
]
fig, ax = plt.subplots(figsize=(9, 3.8))
bars = ax.barh(source_labels, source_plot['missing_pct'])
ax.bar_label(bars, labels=[f'{x:.1f}%' for x in source_plot['missing_pct']], padding=4)
ax.set(
    title='Отсутствие суммы дохода по его источнику',
    xlabel='Клиенты без указанной суммы дохода, %', ylabel='',
    xlim=(0, max(35, source_plot['missing_pct'].max() * 1.3)),
)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# Укрупнённая проверка без смешения сегмента и источника дохода
source_presence = missing_by('income_source_missing', 'income_missing')
source_presence.index = source_presence.index.map({
    False: 'Источник известен', True: 'Источник не указан',
})
display(source_presence.round(1))

Вывод: клиент, которые не указали источник дохода, пропускали сумму дохода в ~5 раз чаще, чем клиенты с зарплатой в другом банке или в нашем банке. 

## 1.7.3 Полнота оценок удовлетворённости 

In [ ]:
score_by_loyalty = missing_by('loyalty_tier', 'score_missing')
loyalty_order = [
    x for x in ['Базовый', 'Серебро', 'Золото', 'Платина']
    if x in score_by_loyalty.index
]
loyalty_order += [x for x in score_by_loyalty.index if x not in loyalty_order]
score_by_loyalty = score_by_loyalty.reindex(loyalty_order)
display(score_by_loyalty.round(1))

loyalty_labels = [
    f'{name} (n={int(row.n_clients)})'
    for name, row in score_by_loyalty.iterrows()
]
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(loyalty_labels, score_by_loyalty['missing_pct'])
ax.bar_label(
    bars, labels=[f'{x:.1f}%' for x in score_by_loyalty['missing_pct']], padding=4,
)
ax.invert_yaxis()
ax.set(
    title='Отсутствие оценки по уровням лояльности',
    xlabel='Клиенты без оценки удовлетворённости, %', ylabel='', xlim=(0, 100),
)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Вывод: Клиенты базового и серебряного уровней в ~2 раза реже оставляют оценки, чем клиенты золотого и платинового уровней 

## 1.7.4 Совместные пропуски


In [ ]:
n_missing = df_clients[flags].sum(axis=1)
completeness = (
    n_missing.value_counts().reindex(range(4), fill_value=0)
    .rename_axis('Отсутствует полей').to_frame('Клиентов')
)
completeness['Доля, %'] = completeness['Клиентов'].div(len(df_clients)).mul(100)
display(completeness.round(2))

print(f'Два или три пропуска: {n_missing.ge(2).sum()} ({n_missing.ge(2).mean():.2%})')
print(f'Все три сведения известны: {n_missing.eq(0).sum()} ({n_missing.eq(0).mean():.2%})')

## Вывод по пропускам
- строки с пропусками не удаляем и не заменяем другими значениями, так как тогда произойдёт смещение. Сохраняем созданные столбцы: флаги пропусков.
- причины пропусков и механизм MAR/MNAR не установлены

 

## 2.  Аудит и подготовка таблицы transactions

## 2.1 Чтение и проверка типов

In [ ]:
# Чтение таблицы транзакций с преобразованием дат и сумм
df_transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    sep=';',
    encoding='cp1251',
    dtype={
        'txn_id': 'str',
        'client_id': 'str',
        'currency': 'category',
        'category': 'category',
        'mcc_code': 'category',
        'channel': 'category',
        'status': 'category'
    },
    parse_dates=['txn_datetime'],
    date_format={'txn_datetime': 'ISO8601'},
    converters={
        'amount': lambda value: float(
            ''.join(value.split()).replace(',', '.')
        ) if value.strip().casefold() not in {
            '', 'nan', 'na', 'n/a', 'null', 'none'
        } else np.nan
    }
)

# Проверяем, что дата действительно преобразована
assert pd.api.types.is_datetime64_any_dtype(
    df_transactions['txn_datetime']
), 'txn_datetime остался строкой: нужно проверить формат исходных дат'

# Проверка результата
print(f"Размер таблицы: {df_transactions.shape}")

print("\nТипы данных:")
print(df_transactions.dtypes)

print("\nПервые 5 строк:")
display(df_transactions.head())

Вывод: таблица transactions содержит 629930 строк и 9 столбцов. Все столбцы прочитаны с нужными типами.

## 2.2 Проверка категориальных значений и их пропусков

In [ ]:
# 1. Анализ пропусков (NaN / NaT) по всем столбцам
missing_stats = pd.DataFrame({
    'Пропуски (шт)': df_transactions.isna().sum(),
    'Пропуски (%)': (df_transactions.isna().sum() / len(df_transactions) * 100).round(2)
})

print("Статистика пропусков в таблице транзакций:")
display(missing_stats)


# 2. Анализ категориальных столбцов
# Автоматически находим все столбцы типов 'category'
cat_cols = df_transactions.select_dtypes(include=['category']).columns.tolist()

for col in cat_cols:
   
    print(f"\n Столбец: '{col}'")
    print(f"   Уникальных значений нормализации: {df_transactions[col].nunique(dropna=False)}")

    # Выводим распределение уже нормализованных значений
    counts = df_transactions[col].value_counts(dropna=False)
    print(counts.to_string()) 

Вывод: Пропусков в категориальных столбцах нет, категориальные столбцы имеют ограниченные количества категорий. TcЕсть часть названий категорий в некоторых столбцах, нужна нормализация:

In [ ]:
# Нормализуем каждый столбец по своему правилу.
# После преобразования возвращаем тип category.
cat_cols = ['currency', 'category', 'mcc_code', 'channel', 'status']

for col in cat_cols:
    values = (
        df_transactions[col]
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

    if col == 'currency':
        # RUB, USD, EUR
        values = values.str.upper()

    elif col == 'status':
        # успешно, отклонено, возврат
        values = values.str.casefold()

    elif col == 'channel':
        # Карта, Токен, Онлайн-банк, СБП
        values = values.str.capitalize().replace({
            'Сбп': 'СБП',
        })

    elif col == 'category':
        values = values.str.capitalize().replace({
            'Кафе/рестораны': 'Кафе и рестораны',
            'Связь и интернет': 'Связь и Интернет',
            'Азс': 'АЗС',
        })

    # Для mcc_code меняем только пробелы, не сам код.
    df_transactions[col] = values.astype('category')

print('Нормализация завершена.')

# Проверяем типы и итоговые распределения.
display(df_transactions[cat_cols].dtypes.to_frame('Тип данных'))

for col in cat_cols:
    print(f"\nСтолбец: {col}")
    print(
        'Уникальных заполненных значений:',
        df_transactions[col].nunique(),
    )
    print('Пропусков:', df_transactions[col].isna().sum())
    print(df_transactions[col].value_counts(dropna=False).to_string())

## 2.3 Сумм, валют, статусов

In [ ]:
amount = df_transactions['amount']

quality = pd.Series({
    'Всего операций': len(df_transactions),
    'Пропусков во всех полях': df_transactions.isna().sum().sum(),
    'Пропусков суммы': amount.isna().sum(),
    'Бесконечных сумм': np.isinf(
        amount.to_numpy(dtype=float, na_value=np.nan)
    ).sum(),
    'Нулевых сумм': amount.eq(0).sum(),
    'Отрицательных сумм': amount.lt(0).sum(),
    'Положительных сумм': amount.gt(0).sum(),
}, name='Количество')

display(quality.to_frame())

Вывод: пропусков, бесконечных сумм и нулевых значений нет, но есть отрицательные суммы, которые надо проверить, например, на согласованность знака суммы и статуса

In [ ]:
amount = df_transactions['amount']
status = df_transactions['status'].astype('string').str.strip().str.casefold()

# Невалидные суммы не должны незаметно исчезнуть из проверки знака.
assert np.isfinite(
    amount.to_numpy(dtype=float, na_value=np.nan)
).all(), 'В amount есть пропуски или бесконечности: сначала обработай их.'

sign = pd.Series(
    np.select(
        [amount.lt(0), amount.eq(0), amount.gt(0)],
        ['Отрицательная', 'Нулевая', 'Положительная'],
        default='Не определён',
    ),
    index=df_transactions.index,
    name='Знак суммы',
)

sign_counts = pd.crosstab(
    status.fillna('Статус отсутствует'),
    sign,
).reindex(
    columns=['Отрицательная', 'Нулевая', 'Положительная'],
    fill_value=0,
)

# Цвет — доля внутри строки; подписи — реальные количества.
sign_pct = sign_counts.div(sign_counts.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots(figsize=(9, 3.8))

sns.heatmap(
    sign_pct,
    annot=sign_counts,
    fmt=',d',
    cmap='Blues',
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={'label': 'Доля внутри статуса, %'},
    ax=ax,
)

ax.set(
    title='Знак суммы и статус операции',
    xlabel='Знак суммы',
    ylabel='Статус',
)
ax.tick_params(axis='both', labelrotation=0)

plt.tight_layout()
plt.show()

is_return = status.eq('возврат').fillna(False)

print(
    'Отрицательных сумм вне возвратов:',
    int((amount.lt(0) & ~is_return).sum()),
)
print(
    'Возвратов с неотрицательной суммой:',
    int((is_return & amount.ge(0)).sum()),
)

Вывод: отрицательные суммы это не ошибки! Возвраты сохраняют со знаком минус, поэтому в будующем для анализа достаточно ориентироваться на статусы операций и трактовать соответсвенно.

In [ ]:
# Распределение сумм успешных операций
successful = df_transactions.loc[
    status.eq('успешно').fillna(False) & amount.gt(0)
].copy()

grouped = successful.groupby('currency', observed=True)['amount']

amount_stats = grouped.agg(
    n='size',
    median='median',
    mean='mean',
    min='min',
    max='max',
)

quantiles = grouped.quantile([0.25, 0.75, 0.99]).unstack()
quantiles.columns = ['q25', 'q75', 'p99']

amount_stats = amount_stats.join(quantiles)
amount_stats['iqr'] = amount_stats['q75'] - amount_stats['q25']

display(
    amount_stats[
        ['n', 'median', 'mean', 'q25', 'q75', 'iqr', 'p99', 'min', 'max']
    ].round(2)
)

Вывод: среднее заметно выше медианы, получаются, что крупные операции формируют правый хвост, поэтому центральный размер покупки лучше описывается медианой.

In [ ]:
# Список валют и их символов для отображения
currencies = [
    ('RUB', '₽'),
    ('USD', '$'),
    ('EUR', '€'),
]

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 12),
    sharex=False,  # оси X независимые, так как диапазоны сумм разные
)

for ax, (currency, symbol) in zip(axes, currencies):
    # Фильтруем успешные операции по валюте
    currency_amount = successful.loc[
        successful['currency'].eq(currency), 'amount'
    ]

    if currency_amount.empty:
        ax.text(
            0.5, 0.5,
            f'Нет успешных операций в {currency}',
            ha='center', va='center',
            transform=ax.transAxes,
            fontsize=12,
        )
        ax.set_title(f'Валюта: {currency}')
        continue

    # Бины для гистограммы (логарифмическая сетка)
    bins = (
        np.geomspace(currency_amount.min(), currency_amount.max(), 45)
        if currency_amount.nunique() > 1
        else [currency_amount.iloc[0] / 1.1, currency_amount.iloc[0] * 1.1]
    )

    # Гистограмма
    ax.hist(
        currency_amount,
        bins=bins,
        color='steelblue',
        edgecolor='white',
    )

    # Линия медианы
    ax.axvline(
        currency_amount.median(),
        color='darkorange',
        linestyle='--',
        label=f'Медиана: {currency_amount.median():,.2f} {symbol}',
    )

    # Линия среднего
    ax.axvline(
        currency_amount.mean(),
        color='crimson',
        linestyle='-.',
        label=f'Среднее: {currency_amount.mean():,.2f} {symbol}',
    )

    # Оформление
    ax.set_xscale('log')
    ax.set(
        title=f'Распределение сумм успешных операций ({currency})',
        xlabel=f'Сумма, {symbol} — логарифмическая шкала',
        ylabel='Количество операций',
    )
    ax.legend(loc='upper right')
    sns.despine()

plt.tight_layout()
plt.show()

Вывод: логарифмическая шкала для удобного отображения. Большие суммы встречаются реже, чем маленькие, то есть распределение смещено(не симметрично)

In [ ]:
# Для сравнения покупок не включаем возвраты и отклонённые попытки.
plot_data = df_transactions.loc[
    df_transactions['status'].astype('string')
    .str.strip().str.casefold().eq('успешно').fillna(False)
    & df_transactions['amount'].gt(0),
    ['currency', 'category', 'amount'],
].copy()

# Объединяем варианты написания категорий в выборке для графиков.
plot_data['category'] = (
    plot_data['category'].astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.capitalize()
    .replace({
        'Кафе/рестораны': 'Кафе и рестораны',
        'Азс': 'АЗС',
        'Связь и интернет': 'Связь и Интернет',
    })
    .fillna('Категория не указана')
)

for currency in ['RUB', 'USD', 'EUR']:
    data = plot_data.loc[plot_data['currency'].eq(currency)]

    if data.empty:
        continue

    # Сортируем категории по типичному размеру покупки.
    category_stats = (
        data.groupby('category', observed=True)['amount']
        .agg(n='size', median='median')
        .sort_values('median')
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    sns.boxplot(
        data=data,
        x='amount',
        y='category',
        order=category_stats.index.tolist(),
        color='steelblue',
        whis=1.5,
        flierprops={
            'marker': 'o',
            'markersize': 3,
            'alpha': 0.35,
            'markerfacecolor': 'none',
        },
        ax=ax,
    )

    # Шкалу меняем после построения:
    # квартильные границы и усы рассчитаны по исходным суммам.
    ax.set_xscale('log')

    ax.set_yticks(
        range(len(category_stats)),
        labels=[
            f'{category} (n={int(n):,})'.replace(',', ' ')
            for category, n in category_stats['n'].items()
        ],
    )

    ax.set(
        title=f'Суммы успешных операций по категориям — {currency}',
        xlabel=f'Сумма, {currency} — логарифмическая шкала',
        ylabel='',
    )

    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.show()